In [0]:
# %sql
# DROP TABLE IF EXISTS workspace.futbol.silver_match_events;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS workspace.futbol.silver_match_events (
    match_id STRING COMMENT 'ID único del partido extraído de la URL',
    event_time TIMESTAMP COMMENT 'Timestamp del evento',
    event_team STRING COMMENT 'Nombre del equipo asociado al evento',
    description STRING COMMENT 'Descripción completa del evento',
    event_type STRING COMMENT 'Tipo de evento categorizado (GOL, TARJETA AMARILLA, TARJETA ROJA, CAMBIO, OTHER)',
    match_date STRING COMMENT 'Fecha del partido',
    match_name STRING COMMENT 'Nombre del partido (equipos)',
    league STRING COMMENT 'Liga del partido',
    processed_at TIMESTAMP COMMENT 'Timestamp de procesamiento a Silver'
)
USING DELTA
COMMENT 'Tabla Silver de eventos de partidos de fútbol'
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true',
    'quality' = 'silver'
)

In [0]:
%sql
-- SELECT
--     COUNT(*) as total_rows,
--     COUNT(DISTINCT match_id) AS total_matchs,
--     COUNT(DISTINCT team_name) AS total_teams,
--     COUNT(DISTINCT player_position) AS total_positions
-- FROM
--     workspace.futbol.bronze_matchs
-- ;

In [0]:
%sql
MERGE INTO workspace.futbol.silver_match_events AS target
USING (
    SELECT DISTINCT
        match_id,
        event_time,
        event_team,
        description,
        event_type,
        match_date,
        match_name,
        league,
        processed_at
    FROM (
        SELECT
            REPLACE(substring_index(get_json_object(match, '$[0].url'), '-', -1), "/", "") AS match_id,
            CAST(get_json_object(sub_event_json, '$.startDate') AS TIMESTAMP) AS event_time,
            DECODE(ENCODE(get_json_object(sub_event_json, '$.attendee.name'), 'ISO-8859-1'), 'UTF-8') AS event_team,
            DECODE(ENCODE(get_json_object(sub_event_json, '$.name'), 'ISO-8859-1'), 'UTF-8') AS description,
            CASE
                WHEN get_json_object(sub_event_json, '$.name') IS NULL THEN 'OTHER'
                WHEN get_json_object(sub_event_json, '$.name') LIKE 'Gol%' THEN 'GOL'
                WHEN get_json_object(sub_event_json, '$.name') LIKE 'Amonest%' THEN 'TARJETA AMARILLA'
                WHEN get_json_object(sub_event_json, '$.name') LIKE 'Expulsi%' THEN 'TARJETA ROJA'
                WHEN get_json_object(sub_event_json, '$.name') LIKE 'Sale%' THEN 'CAMBIO'
                ELSE 'OTHER'
            END AS event_type,
            date AS match_date,
            DECODE(ENCODE(name, 'ISO-8859-1'), 'UTF-8') AS match_name,
            DECODE(ENCODE(league, 'ISO-8859-1'), 'UTF-8') AS league,
            CURRENT_TIMESTAMP() AS processed_at
        FROM 
            workspace.futbol.bronze_matchs
        LATERAL VIEW EXPLODE(from_json(get_json_object(match, '$[0].subEvent'), 'array<string>')) exploded AS sub_event_json
        WHERE get_json_object(match, '$[0].subEvent') IS NOT NULL
    )
) AS source
ON target.match_id = source.match_id 
    AND target.event_time = source.event_time 
    AND COALESCE(target.event_team, '') = COALESCE(source.event_team, '')
    AND target.description = source.description
WHEN MATCHED THEN
    UPDATE SET
        target.event_type = source.event_type,
        target.match_date = source.match_date,
        target.match_name = source.match_name,
        target.league = source.league,
        target.processed_at = source.processed_at
WHEN NOT MATCHED THEN
    INSERT (
        match_id, event_time, event_team, description, event_type,
        match_date, match_name, league, processed_at
    )
    VALUES (
        source.match_id, source.event_time, source.event_team, source.description,
        source.event_type, source.match_date, source.match_name, source.league,
        source.processed_at
    )

In [0]:
%sql
-- Distribución de eventos por tipo
SELECT 
    event_type,
    COUNT(*) AS event_count,
    ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER(), 2) AS percentage
FROM workspace.futbol.silver_match_events
GROUP BY event_type
ORDER BY event_count DESC

In [0]:
%sql
-- Top 10 ligas con más eventos registrados
SELECT 
    league,
    COUNT(DISTINCT match_id) AS total_matches,
    COUNT(*) AS total_events,
    ROUND(COUNT(*) * 1.0 / COUNT(DISTINCT match_id), 2) AS avg_events_per_match
FROM workspace.futbol.silver_match_events
GROUP BY league
ORDER BY total_events DESC
;

In [0]:
%sql
-- Últimos 10 eventos registrados
SELECT 
    match_name,
    league,
    event_time,
    event_team,
    event_type,
    description
FROM workspace.futbol.silver_match_events
ORDER BY event_time DESC
LIMIT 10

In [0]:
# %sql
# SELECT
# *
# FROM
# workspace.futbol.silver_match_events
# DESC
# LIMIT 5;